In [2]:
sample = 'mouse_skin'
save_dir = 'barcode'

In [ ]:
import cv2
import glob
import numpy as np
import pandas as pd
import scipy.io
from scipy.ndimage import rotate
from skimage import transform as tf
from skimage.transform import warp
import scipy.io
import cv2
import matplotlib.pyplot as plt
import os

all_cell_mapping = pd.read_csv(f'./{sample}/all_cell_mapping.csv', index_col=0)
all_cell_raman = pd.read_csv(f'./{sample}/all_cell_raman.csv', index_col=0)

In [ ]:

# Function to generate distinct colors from the 'jet' colormap
def generate_tab20_extended_colors():
    colors = list(plt.cm.tab20(np.linspace(0, 1, 20)))  # Generate 20 colors from tab20
    additional_color = plt.cm.Set2(5)  # Select a color from another colormap, e.g., the second color in 'Set1'
    colors.append(additional_color)  # Append the additional color to the list
    return colors

# Generate 21 colors
extended_colors = generate_tab20_extended_colors()
# define each cell type a color
color_mapping = {cell_type: color for cell_type, color in zip(all_cell_mapping['cell_type'].unique(), extended_colors)}
print(color_mapping)

In [ ]:
from scipy.integrate import simpson
from scipy.stats import ranksums
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import numpy as np

wave_number = scipy.io.loadmat('mouse_skin/wavenumbers.mat')['wavenumber'][0][413:1286].round(0).astype(np.int16)

def normalize_spectra(wavenum, intensity, min_peak, max_peak):
    amide_mask = (wavenum >= min_peak) & (wavenum <= max_peak)
    area = simpson(intensity[amide_mask], x=wavenum[amide_mask])
    # print(area)
    return intensity / area

def plot_barcode_comparison(features, group1_values, group2_values, label_set, cell_type, prefix):
   
    sorted_indices = np.argsort(group1_values)
    sorted_features = [features[i] for i in sorted_indices]
    print(sorted_features)
    # print(group2_values)
    sorted_indices = np.argsort(group2_values) # from small to large
    sorted_features = [features[i] for i in sorted_indices]
    print(sorted_features)

    # Create a figure and axis
    fig, ax = plt.subplots(figsize=(10, 3))

    # Plot the first array as a barcode
    for i in range(len(group1_values)):
        ax.plot([group1_values[i], group1_values[i]], [0, 1], color='#91B9D6', linewidth=0.5, alpha=0.8)
        # plot text
        # ax.text(group1_values[i], 1.2, features[i], ha='center', va='center', fontsize=1, color='black')
        # rotate text
        ax.text(group1_values[i], 1.2, features[i], ha='center', va='center', fontsize=1, color='black', rotation=90)
       

    # Plot the second array as a barcode (offset the lines slightly to avoid overlap)
    for i in range(len(group2_values)):
        ax.plot([group2_values[i], group2_values[i]], [1.5, 2.5], color='#91B9D6', linewidth=0.5, alpha=0.8)
        # plot text
        ax.text(group2_values[i], 3, features[i], ha='center', va='center', fontsize=1, color='black', rotation=90)
      

    # Add labels and title
    ax.set_yticks([0.5, 2.0])

    if prefix.startswith('Old'):
        ax.set_yticklabels(['Senescence', 'Non-senescence'])
    else:
        ax.set_yticklabels(['Old', 'Young'])

    # Remove the spines
    plt.gca().spines['left'].set_visible(False)
    plt.gca().spines['right'].set_visible(False)
    plt.gca().spines['top'].set_visible(False)

    plt.grid(False)
    # plt.legend()
    plt.title(f'{label_set} ({cell_type})')  # Add a title

    if '/' in cell_type:
        cell_type = cell_type.replace('/', '_')


    # tight layout
    plt.tight_layout()

    save_path = 'figures' + '/' + sample + '/' + save_dir
    if not os.path.exists(save_path):
        os.makedirs(save_path)


    plt.savefig(f'{save_path}/{prefix}_{label_set}_{cell_type}.pdf')
    plt.show()
    

def calculate_DEP(data1, data2, wave_number, label_set, cell_type, prefix):

    import numpy as np
    import pandas as pd
    import scanpy as sc

    data1 = pd.DataFrame(data1, columns=wave_number) # positive
    data2 = pd.DataFrame(data2, columns=wave_number)

    # Calculate the Wilcoxon rank-sum test for each feature (column)
    p_values = []
    log2FC = []
    for column in data1.columns:
        stat, p_value = ranksums(data1[column], data2[column])
        p_values.append(p_value)

        mean_data1 = data1[column].mean()
        mean_data2 = data2[column].mean()
        log2FC.append(np.log2(mean_data1 / mean_data2))
        
    # Apply FDR correction
    _, p_values_corrected, _, _ = multipletests(p_values, alpha=0.05, method='fdr_bh')


    # Create a DataFrame with the original and corrected p-values
    results = pd.DataFrame({
            'Index': range(len(data1.columns)),
            'Peak': data1.columns,
            'p-value': p_values,
            'p-value_corrected': p_values_corrected,
            'log2FC': log2FC
        })

    # Filter features with FDR < 0.05
    significant_results = results[results['p-value_corrected'] <= 0.05]


    upregulated = significant_results[significant_results['log2FC'] > 0]
    downregulated = significant_results[significant_results['log2FC'] < 0]

    upregulated = upregulated.sort_values('log2FC', ascending=False)
    downregulated = downregulated.sort_values('log2FC', ascending=True)
    

    print('Number of DEPs:', len(significant_results), 'upregulated:', len(upregulated), 'downregulated:', len(downregulated))

    if len(upregulated) < 1 or len(downregulated) < 1:
        return None
     
    # Print significant features
    # print('Number of DEFs:', len(significant_results), 'upregulated:', len(upregulated), 'downregulated:', len(downregulated))
    else:

        # plot_volcano

        print(label_set, cell_type, len(upregulated), len(downregulated))
       
        # rank significant_results by log2FC
        significant_results = significant_results.sort_values('log2FC', ascending=True)

        selected_peaks = significant_results['Peak'].values
        selected_wavenum = [wave_number.tolist().index(peak) for peak in selected_peaks]
        
        if len(upregulated) < 30:
            number_of_selected_up_peaks = len(upregulated)
        else:
            number_of_selected_up_peaks = 30
        if len(downregulated) < 30:
            number_of_selected_down_peaks = len(downregulated)
        else:
            number_of_selected_down_peaks = 30
     
        # selected_peaks = np.concatenate([selected_peaks[:number_of_selected_peaks], selected_peaks[-number_of_selected_peaks:]])
        selected_peaks = np.concatenate([selected_peaks[:number_of_selected_down_peaks], selected_peaks[-number_of_selected_up_peaks:]])
        # selected_wavenum = np.concatenate([selected_wavenum[:number_of_selected_peaks], selected_wavenum[-number_of_selected_peaks:]])

        print(selected_peaks)


        mean_data1 = data1.mean(0)
        mean_data2 = data2.mean(0)

        mean_data1 = mean_data1[selected_peaks]
        mean_data2 = mean_data2[selected_peaks]

        print(mean_data1.values, mean_data2.values)

        plot_barcode_comparison(selected_peaks, mean_data1.values, mean_data2.values, label_set, cell_type, prefix)


In [ ]:
from scipy import stats

label_sets = ['old']
tiltle_sets = ['old']

# global
for label_set,tiletle_set in zip(label_sets, tiltle_sets):

    selected_cell_mapping = all_cell_mapping[all_cell_mapping['sample_type'] == 'O'] # select only old cells

    if len(selected_cell_mapping) < 1:
        continue

    selected_cell_raman = all_cell_raman.loc[selected_cell_mapping.index]

    labels = selected_cell_mapping['p21+'].values * 1
    features = selected_cell_raman.values.astype(np.float32)
    
    new_features = []
    for i in range(features.shape[0]):
        new_features.append(normalize_spectra(wave_number, features[i], 1630, 1700))
    features = np.stack(new_features, axis=0)

    positive = features[labels == 1]
    negative = features[labels == 0]
    
    if len(positive) < 1 and len(negative) < 1:
        continue

    calculate_DEP(positive, negative, wave_number, label_set, 'global', 'SvsNS')


# cell type
for cell_type in all_cell_mapping['cell_type'].unique():

    selected_cell_mapping = all_cell_mapping[all_cell_mapping['sample_type'] == 'O'] # select only old cells
    selected_cell_type_mapping = selected_cell_mapping[selected_cell_mapping['cell_type'] == cell_type]

    if len(selected_cell_type_mapping) < 1:
        continue

    selected_cell_raman = all_cell_raman.loc[selected_cell_type_mapping.index]

    labels = selected_cell_type_mapping['p21+'].values * 1
    features = selected_cell_raman.values.astype(np.float32)

    new_features = []
    for i in range(features.shape[0]):
        new_features.append(normalize_spectra(wave_number, features[i], 1630, 1700))
    features = np.stack(new_features, axis=0)

    positive = features[labels == 1]
    negative = features[labels == 0]

    if len(positive) < 1 and len(negative) < 1:
        continue

    calculate_DEP(positive, negative, wave_number, label_sets[0], cell_type, 'SvsNS')
